# 03 — Advanced models, ensemble and drift (Week 3, v0.2)

**Modules 3–4** — owner: Basile Chabrel, reviewer: Augustin Nilsson

Same table, same split, same metric and same rules as notebook 02 (selection on validation, test touched once at the end).

**Limits of v0.1 that motivate v0.2**
1. The best v0.1 model is linear: it cannot learn interactions (e.g. "high scoring *and* winning team").
2. On test it beats the "same as last year" baseline by ~5 PR-AUC points, but the confidence intervals overlap.
3. It misses players with no previous season and breakout players.

In [1]:
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import average_precision_score

from src.config import FIGURE_DPI, FIGURES_DIR, SEED, TABLES_DIR, TARGET_COL, ensure_dirs
from src.features.build_dataset import build_model_table
from src.models.split import temporal_split, feature_columns
from src.models.automl_v01 import baseline_scores, candidate_grids, run_automl, score_of
from src.models.advanced_v02 import advanced_grids, permutation_importance_table, rank_average
from src.evaluation.metrics import best_f1_threshold, hits_at_k, point_metrics, season_bootstrap_ci

ensure_dirs()
table = build_model_table("prev")
parts = temporal_split(table)
train, validation, test = parts["train"], parts["validation"], parts["test"]
cols = feature_columns(table)

## 1. Model selection on validation: v0.1 + v0.2 candidates

- **Random forest** and **histogram gradient boosting**: advanced, ensemble versions of the decision tree seen in class.
- **MLP** (multi-layer perceptron): the algorithm outside the course. Reference: Grinsztajn, Oyallon & Varoquaux, *Why do tree-based models still outperform deep learning on typical tabular data?*, NeurIPS 2022 (Datasets & Benchmarks), arXiv:2207.08815. They show tree ensembles stay state-of-the-art on medium-sized tabular data (~10K rows); our training set has 11.5K rows, so the **hypothesis** is that the MLP will not beat gradient boosting. We test it rather than assume it.

In [2]:
grids = {**candidate_grids(), **advanced_grids()}
selection = run_automl(train, validation, grids)
models = selection.attrs["searches"]
print(selection.to_string(index=False))

              model                                                                                                    best_params  validation_pr_auc
                mlp                                                   {'model__alpha': 1.0, 'model__hidden_layer_sizes': (64, 32)}           0.758274
logistic_regression                                                                         {'model__C': 0.01, 'select__k': 'all'}           0.753260
  gradient_boosting {'model__l2_regularization': 0.0, 'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__max_iter': 200}           0.741410
                knn                                  {'model__n_neighbors': 150, 'model__weights': 'distance', 'select__k': 'all'}           0.734405
      random_forest                                                       {'model__max_depth': None, 'model__min_samples_leaf': 5}           0.733277
      decision_tree                                     {'model__max_depth': 8, 'model__min_samples_

## 2. Ensemble: rank average of the three best models

Soft voting on **ranks** (not raw scores, which live on different scales). The members are the three best models on validation; the ensemble is then compared on validation like any other candidate.

In [3]:
top3 = selection.model.head(3).tolist()
val_scores = {m: score_of(models[m], validation[cols]) for m in models}
val_scores["ensemble_top3"] = rank_average([val_scores[m] for m in top3])
val_table = pd.DataFrame({
    m: point_metrics(validation[["season", TARGET_COL]].assign(score=s), best_f1_threshold(validation[TARGET_COL], s))
    for m, s in val_scores.items()}).T.sort_values("pr_auc", ascending=False)
print("Ensemble members:", top3)
print(val_table[["pr_auc", "roc_auc", "f1", "hits_at_k"]].round(3).to_string())
val_table.round(4).to_csv(TABLES_DIR / "week3_validation_all_models.csv")
final_name = val_table.index[0]
print(f"\nSelected on validation: {final_name}")

Ensemble members: ['mlp', 'logistic_regression', 'gradient_boosting']
                     pr_auc  roc_auc     f1  hits_at_k
mlp                   0.758    0.974  0.707      0.682
ensemble_top3         0.757    0.974  0.711      0.693
logistic_regression   0.753    0.975  0.695      0.676
gradient_boosting     0.741    0.970  0.700      0.693
knn                   0.734    0.973  0.702      0.693
random_forest         0.733    0.973  0.679      0.659
decision_tree         0.666    0.949  0.655      0.642
svm_rbf               0.661    0.965  0.634      0.637

Selected on validation: mlp


## 3. Final evaluation on test (2018–2026), once

Every model is reported for transparency, but the **selection was made in section 2 on validation**; picking the best test line afterwards would be test-set overfitting.

In [4]:
def test_scores(name):
    if name == "ensemble_top3":
        return rank_average([score_of(models[m], test[cols]) for m in top3])
    return score_of(models[name], test[cols])

rows, cis = {}, {}
for name in val_table.index:
    s_test = test_scores(name)
    thr = best_f1_threshold(validation[TARGET_COL], val_scores[name])  # threshold from validation
    frame = test[["season", TARGET_COL]].assign(score=s_test)
    res = season_bootstrap_ci(frame, thr, n_boot=500)
    rows[name] = res["value"]
    cis[name] = res.apply(lambda r: f"[{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
for name, s in baseline_scores(test).items():
    if name != "random":
        frame = test[["season", TARGET_COL]].assign(score=s)
        res = season_bootstrap_ci(frame, best_f1_threshold(validation[TARGET_COL], baseline_scores(validation)[name]), n_boot=500)
        rows[f"baseline: {name}"], cis[f"baseline: {name}"] = res["value"], res.apply(lambda r: f"[{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
test_table = pd.DataFrame(rows).T[["pr_auc", "roc_auc", "precision", "recall", "f1", "hits_at_k"]]
test_ci = pd.DataFrame(cis).T[["pr_auc", "hits_at_k"]]
print(test_table.round(3).to_string()); print(); print(test_ci.to_string())
test_table.round(4).to_csv(TABLES_DIR / "week3_test_all_models.csv"); test_ci.to_csv(TABLES_DIR / "week3_test_ci.csv")

                             pr_auc  roc_auc  precision  recall     f1  hits_at_k
mlp                           0.679    0.975      0.562   0.697  0.622      0.602
ensemble_top3                 0.683    0.975      0.607   0.622  0.615      0.606
logistic_regression           0.678    0.975      0.540   0.734  0.622      0.614
gradient_boosting             0.665    0.966      0.611   0.606  0.608      0.606
knn                           0.666    0.971      0.602   0.639  0.620      0.602
random_forest                 0.668    0.966      0.561   0.685  0.617      0.593
decision_tree                 0.628    0.954      0.621   0.577  0.598      0.598
svm_rbf                       0.604    0.968      0.579   0.672  0.622      0.598
baseline: prev_points         0.628    0.961      0.425   0.817  0.560      0.602
baseline: same_as_last_year   0.631    0.963      0.610   0.564  0.586      0.614

                                     pr_auc       hits_at_k
mlp                          [0.640, 

In [5]:
order = test_table.sort_values("pr_auc").index
fig, ax = plt.subplots(figsize=(8, 4.5))
lo = test_ci.loc[order, "pr_auc"].str.strip("[]").str.split(", ").apply(lambda v: float(v[0]))
hi = test_ci.loc[order, "pr_auc"].str.strip("[]").str.split(", ").apply(lambda v: float(v[1]))
vals = test_table.loc[order, "pr_auc"]
colors = ["#e07a1f" if n == final_name else ("#9aa5b1" if n.startswith("baseline") else "#2a6f97") for n in order]
ax.barh(order, vals, xerr=[vals - lo, hi - vals], color=colors, capsize=3)
ax.axvline(test[TARGET_COL].mean(), color="grey", ls=":")
ax.text(test[TARGET_COL].mean() + 0.01, -0.9, f"random = {test[TARGET_COL].mean():.3f}", color="grey", fontsize=8)
ax.set_xlabel("PR-AUC on test seasons 2018-2026 (95% season-bootstrap CI)")
ax.set_title("Models vs baselines (orange = selected on validation, grey = baselines)")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig7_models_test_pr_auc.png", dpi=FIGURE_DPI); plt.show()

## 4. Which features drive the selected model?

Permutation importance on **validation**: how much PR-AUC drops when one feature is shuffled. Computed on the best single model (an ensemble has no single feature list).

In [6]:
single_best = next(m for m in val_table.index if m != "ensemble_top3")
imp = permutation_importance_table(models[single_best], validation[cols], validation[TARGET_COL], n_repeats=5)
print(f"Model: {single_best}"); print(imp.head(12).round(4).to_string(index=False))
imp.round(5).to_csv(TABLES_DIR / "week3_permutation_importance.csv", index=False)
top = imp.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(top.feature, top.importance_mean, xerr=top.importance_std, color="#2a6f97")
ax.set_xlabel("Drop in validation PR-AUC when shuffled"); ax.set_title(f"Permutation importance — {single_best}")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig8_permutation_importance.png", dpi=FIGURE_DPI); plt.show()

Model: mlp
             feature  importance_mean  importance_std
     n_prev_all_star           0.0579          0.0086
                 age           0.0235          0.0033
   prev_pts_per_game           0.0168          0.0041
   prev_ast_per_game           0.0153          0.0024
              prev_g           0.0147          0.0013
   prev_tov_per_game           0.0122          0.0031
all_star_prev_season           0.0122          0.0059
          experience           0.0116          0.0032
   prev_team_win_pct           0.0113          0.0030
             prev_ws           0.0103          0.0042
   prev_fga_per_game           0.0094          0.0029
           prev_dbpm           0.0084          0.0020


## 5. Drift check

Does an old model still work today? Same algorithm, trained on two windows, both evaluated on the test seasons.

In [7]:
best_single = models[single_best]
windows = {"trained 1981-2000": table[table.season <= 2000],
           "trained 1981-2017": table[table.season <= 2017],
           "trained 2001-2017 (recent only)": table[table.season.between(2001, 2017)]}
drift = {}
for label, data in windows.items():
    m = clone(best_single).fit(data[cols], data[TARGET_COL])
    s = score_of(m, test[cols])
    drift[label] = {"test_pr_auc": average_precision_score(test[TARGET_COL], s),
                    "test_hits_at_k": point_metrics(test[["season", TARGET_COL]].assign(score=s), 0.5)["hits_at_k"]}
drift = pd.DataFrame(drift).T
print(drift.round(3).to_string()); drift.round(4).to_csv(TABLES_DIR / "week3_drift.csv")

per_season = (test[["season", TARGET_COL]].assign(score=test_scores(final_name))
              .groupby("season").apply(lambda g: average_precision_score(g[TARGET_COL], g.score), include_groups=False))
print("\nPR-AUC per test season (selected model):"); print(per_season.round(3).to_string())

                                 test_pr_auc  test_hits_at_k
trained 1981-2000                      0.668           0.618
trained 1981-2017                      0.686           0.602
trained 2001-2017 (recent only)        0.678           0.602

PR-AUC per test season (selected model):
season
2018    0.771
2019    0.737
2020    0.672
2021    0.734
2022    0.680
2023    0.620
2024    0.763
2025    0.614
2026    0.627


## 6. Case study: the 2026 All-Star selection

Top-K of the selected model for 2026 (K = 28 real All-Stars), predicted *before* the season started.

In [8]:
s26 = test[test.season == 2026][["player", TARGET_COL, "prev_pts_per_game", "n_prev_all_star"]].copy()
s26["score"] = test_scores(final_name)[test.season.to_numpy() == 2026]
k = int(s26[TARGET_COL].sum())
s26 = s26.sort_values("score", ascending=False).reset_index(drop=True)
top_k = s26.head(k)
print(f"2026: {int(top_k[TARGET_COL].sum())} of {k} All-Stars found in the model's top {k}")
print("\nPredicted but NOT selected (false positives):")
print(top_k[top_k[TARGET_COL] == 0][["player", "prev_pts_per_game", "n_prev_all_star"]].to_string(index=False))
missed = s26[(s26[TARGET_COL] == 1) & (s26.index >= k)].assign(rank=lambda d: d.index + 1)
print("\nSelected but missed by the model (rank in the model's list):")
print(missed[["player", "rank", "prev_pts_per_game", "n_prev_all_star"]].to_string(index=False))
s26.head(40).to_csv(TABLES_DIR / "week3_case_2026.csv", index=False)

2026: 15 of 28 All-Stars found in the model's top 28

Predicted but NOT selected (false positives):
          player  prev_pts_per_game  n_prev_all_star
   Anthony Davis               24.7             10.0
    Jayson Tatum               26.8              6.0
    James Harden               22.8             11.0
     Evan Mobley               18.5              1.0
      Trae Young               24.2              4.0
  Jalen Williams               21.6              1.0
  Darius Garland               20.6              2.0
 Zion Williamson               24.6              2.0
     Joel Embiid               23.8              7.0
  Paolo Banchero               25.9              1.0
Domantas Sabonis               19.1              3.0
       Ja Morant               23.2              2.0
     Tyler Herro               23.9              1.0

Selected but missed by the model (rank in the model's list):
        player  rank  prev_pts_per_game  n_prev_all_star
  Devin Booker    31               25.6

## 7. Demonstration: ranking for the 2027 All-Star Game (season in progress)

The model is refitted on all labelled seasons (1981–2026), with the hyperparameters chosen on validation, then scores the players of 2026-27 from their 2025-26 statistics. Only the ranking is shown (the scores are not calibrated probabilities). **This is a forecast, not a result: it cannot be checked before the selections (late January 2027).**

In [9]:
full = table
current = build_model_table("prev", unlabelled_season=2027)
if final_name == "ensemble_top3":
    preds = rank_average([score_of(clone(models[m]).fit(full[cols], full[TARGET_COL]), current[cols]) for m in top3])
else:
    preds = score_of(clone(models[final_name]).fit(full[cols], full[TARGET_COL]), current[cols])
ranking = (current[["player", "prev_pts_per_game", "n_prev_all_star"]].assign(score=preds)
           .sort_values("score", ascending=False).head(24).reset_index(drop=True))
ranking.index += 1
print(ranking.round(3).to_string())
ranking.to_csv(TABLES_DIR / "week3_forecast_2027_top24.csv")

                     player  prev_pts_per_game  n_prev_all_star  score
1              Nikola Jokić               27.7              8.0  0.915
2   Shai Gilgeous-Alexander               31.1              4.0  0.893
3               Luka Dončić               33.5              6.0  0.884
4         Victor Wembanyama               25.0              2.0  0.848
5     Giannis Antetokounmpo               27.6             10.0  0.785
6              Kevin Durant               26.0             16.0  0.772
7           Cade Cunningham               23.9              2.0  0.752
8          Donovan Mitchell               27.9              7.0  0.727
9           Anthony Edwards               28.8              4.0  0.712
10             LeBron James               20.9             22.0  0.711
11             Jaylen Brown               28.7              5.0  0.702
12              Jalen Duren               19.5              1.0  0.672
13             Tyrese Maxey               28.3              2.0  0.666
14    

## 8. Conclusions for the report (to be checked and rewritten by the team)

- Read every number above from the executed outputs, not from this text.
- Compare the selected model with the strongest baseline ("same as last year") **including the confidence intervals**; if they overlap, say so: the gain is not statistically clear.
- The MLP result is the test of the Grinsztajn et al. hypothesis: state whether it held.
- The drift table tells whether old seasons still help or hurt.